In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/final_csv/원본 파일/Chicago_Crimes_2012_to_2017.csv', on_bad_lines='skip')

In [ ]:
df = df.dropna(subset=['X Coordinate','Y Coordinate','Latitude', 'Longitude', 'Location'])

In [ ]:
df.isna().sum()

,0
Unnamed: 0,0
ID,0
Case Number,0
Date,0
Block,0
IUCR,0
Primary Type,0
Description,0
Location Description,1226
Arrest,0


In [ ]:
#  범죄 데이터의 위도, 경도 정보를 사용해 GeoDataFrame 생성
df['geometry'] = df.apply(lambda row: Point(row['Longitude'], row['Latitude']), axis=1)
crime_gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")  # WGS84 좌표계 설정

In [ ]:
# Community Area CSV 파일 불러오기 (the_geom을 WKT 형식으로 변환)
community_areas = pd.read_csv('/content/drive/MyDrive/final_csv/CommAreas_20250325.csv')  # Community Area 데이터 (CSV)
community_areas['geometry'] = community_areas['the_geom'].apply(wkt.loads)  # the_geom을 WKT로 변환
community_areas_gdf = gpd.GeoDataFrame(community_areas, geometry='geometry', crs="EPSG:4326")

In [ ]:
#  2003~2015년 Ward 데이터 불러오기
wards_2003_2015 = pd.read_csv('/content/drive/MyDrive/final_csv/Wards_2003_2015.csv')
wards_2003_2015['geometry'] = wards_2003_2015['the_geom'].apply(wkt.loads)
wards_2003_2015_gdf = gpd.GeoDataFrame(wards_2003_2015, geometry='geometry', crs="EPSG:4326")

In [ ]:
wards_2003_2015_gdf['WARD'].unique()

array(['4', '33', '49', '37', 'OUT', '18', '31', '25', '8', '26', '28',
       '3', '47', '1', '38', '11', '30', '39', '12', '9', '6', '5', '19',
       '41', '23', '24', '46', '44', '36', '48', '27', '50', '7', '15',
       '34', '40', '10', '2', '22', '35', '32', '17', '21', '16', '45',
       '42', '13', '14', '43', '29', '20'], dtype=object)

In [ ]:
wards_2003_2015_gdf = wards_2003_2015_gdf[wards_2003_2015_gdf['WARD'] != 'OUT']
wards_2003_2015_gdf= wards_2003_2015_gdf.dropna(subset=['WARD'])

# int로 변환 후 float로 변환
wards_2003_2015_gdf['WARD'] = wards_2003_2015_gdf['WARD'].astype(float)

In [ ]:
#  2015년 이후 Ward 데이터 불러오기
wards_2015 = pd.read_csv('/content/drive/MyDrive/final_csv/WARDS_2015_20250325.csv')
wards_2015['geometry'] = wards_2015['the_geom'].apply(wkt.loads)
wards_2015_gdf = gpd.GeoDataFrame(wards_2015, geometry='geometry', crs="EPSG:4326")

In [ ]:
wards_2015_gdf['WARD'].unique()

array([12, 16, 15, 20, 49, 23, 29, 14,  3,  4,  2, 35, 21, 24, 13, 48, 31,
       47, 38, 33, 30, 34, 28, 40, 44, 25, 50, 22, 41, 18, 17,  6,  5, 43,
        8, 42,  7, 39, 46, 32,  1, 19,  9, 36, 37, 27, 10, 11, 26, 45])

In [ ]:
# int로 변환 후 float로 변환
wards_2015_gdf['WARD'] = wards_2015_gdf['WARD'].astype(float)

In [ ]:
#  범죄 데이터를 두 개의 기간으로 분리 (2015년 기준)
crime_before_2015 = crime_gdf[crime_gdf['Date'] < '2015-01-01']
crime_after_2015 = crime_gdf[crime_gdf['Date'] >= '2015-01-01']

In [ ]:
wards_2003_2015_gdf.head()

,the_geom,DATA_ADMIN,PERIMETER,WARD,ALDERMAN,CLASS,WARD_PHONE,HALL_PHONE,HALL_OFFIC,ADDRESS,EDIT_DATE1,SHAPE_AREA,SHAPE_LEN,geometry
0,MULTIPOLYGON (((-87.61720985189288 41.84565509...,9.588452e+07,71516.462726,4.0,WILLIAM BURNS,3.0,773-536-8103,312-744-2690,"121 N LASALLE ST, RM 300 OFFICE 10, 60602",4659 S COTTAGE GROVE STE 203,20030527,9.693978e+07,73428.701824,"MULTIPOLYGON (((-87.61721 41.84566, -87.61692 ..."
1,MULTIPOLYGON (((-87.69441592800892 41.95563904...,6.278870e+07,46189.174036,33.0,RICHARD F. MELL,3.0,773-478-8040,312-744-6825,"121 N LASALLE ST, RM 208, 60602",3649 N KEDZIE AV,20020301,6.278870e+07,46189.173373,"MULTIPOLYGON (((-87.69442 41.95564, -87.69443 ..."
2,MULTIPOLYGON (((-87.6642015105961 42.021260262...,4.685230e+07,44816.944176,49.0,JOSEPH A. MOORE,5.0,773-338-5796,312-744-3067,"121 N LASALLE ST, RM 300 OFFICE 24, 60602",7356 N GREENVIEW AV,20030527,4.682849e+07,45091.156684,"MULTIPOLYGON (((-87.6642 42.02126, -87.66419 4..."
3,MULTIPOLYGON (((-87.77086308398384 41.92416619...,8.526330e+07,60411.795941,37.0,EMMA MITTS,2.0,773-745-2894,312-744-8019,"121 N. LASALLE ST, RM 300, 60602",5344 W NORTH AVE,20030527,8.526330e+07,60411.795499,"MULTIPOLYGON (((-87.77086 41.92417, -87.77085 ..."
5,MULTIPOLYGON (((-87.69312171100901 41.76823092...,1.766406e+08,69225.366759,18.0,LONA LANE,5.0,773-471-1991,312-744-6856,"121 N La Salle St, RM 300",8146 S KEDZIE AVE,02-07-07,1.766406e+08,69225.370710,"MULTIPOLYGON (((-87.69312 41.76823, -87.69312 ..."


In [ ]:
#  2003~2015년 데이터에 대한 Spatial Join (Ward 매핑)
crime_before_2015 = gpd.sjoin(crime_before_2015, wards_2003_2015_gdf[['WARD', 'geometry']], how='left', predicate='within')

In [ ]:
#  2015년 이후 데이터에 대한 Spatial Join (Ward 매핑)
crime_after_2015 = gpd.sjoin(crime_after_2015, wards_2015_gdf[['WARD', 'geometry']], how='left', predicate='within')

In [ ]:
#  두 개의 데이터 병합
crime_with_ward = pd.concat([crime_before_2015, crime_after_2015])

In [ ]:
community_areas_gdf.columns

Index(['the_geom', 'PERIMETER', 'AREA', 'COMAREA_', 'COMAREA_ID', 'AREA_NUMBE',
       'COMMUNITY', 'AREA_NUM_1', 'SHAPE_AREA', 'SHAPE_LEN', 'geometry'],
      dtype='object')

In [ ]:
#
crime_with_community = gpd.sjoin(
    crime_with_ward,
    community_areas_gdf[['AREA_NUMBE', 'geometry']],
    how='left',
    predicate='within',
    lsuffix='_ward',
    rsuffix='_community'
)  # Community Area 매핑


In [ ]:
#  'Ward'와 'Community Area' 결측치 보완
crime_with_community['Ward'] = crime_with_community['Ward'].fillna(crime_with_community['WARD'])
crime_with_community['Community Area'] = crime_with_community['Community Area'].fillna(crime_with_community['AREA_NUMBE'])


In [ ]:
#  불필요한 열 삭제 (매핑된 'WARD', 'COMMUNITY' 열 제거)
crime_with_community = crime_with_community.drop(columns=['WARD', 'AREA_NUMBE','index_right','index__community'])

In [ ]:
crime_with_community = crime_with_community.drop(columns=['geometry'])

In [ ]:
#  결측치 확인
print(crime_with_community[['Ward', 'Community Area']].isna().sum())  # 결측치 확인

Ward              10
Community Area     3
dtype: int64


In [ ]:
crime_with_community.isna().sum()

,0
Unnamed: 0,0
ID,0
Case Number,0
Date,0
Block,0
IUCR,0
Primary Type,0
Description,0
Location Description,1226
Arrest,0


In [ ]:
crime_with_community.head()

,Unnamed: 0,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,3,10508693,HZ250496,05/03/2016 11:40:00 PM,013XX S SAWYER AVE,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,True,...,24.0,29.0,08B,1154907.0,1893681.0,2016,05/10/2016 03:56:50 PM,41.864073,-87.706819,"(41.864073157, -87.706818608)"
1,89,10508695,HZ250409,05/03/2016 09:40:00 PM,061XX S DREXEL AVE,0486,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,False,...,20.0,42.0,08B,1183066.0,1864330.0,2016,05/10/2016 03:56:50 PM,41.782922,-87.604363,"(41.782921527, -87.60436317)"
2,197,10508697,HZ250503,05/03/2016 11:31:00 PM,053XX W CHICAGO AVE,0470,PUBLIC PEACE VIOLATION,RECKLESS CONDUCT,STREET,False,...,37.0,25.0,24,1140789.0,1904819.0,2016,05/10/2016 03:56:50 PM,41.894908,-87.758372,"(41.894908283, -87.758371958)"
3,673,10508698,HZ250424,05/03/2016 10:10:00 PM,049XX W FULTON ST,0460,BATTERY,SIMPLE,SIDEWALK,False,...,28.0,25.0,08B,1143223.0,1901475.0,2016,05/10/2016 03:56:50 PM,41.885687,-87.749516,"(41.885686845, -87.749515983)"
4,911,10508699,HZ250455,05/03/2016 10:00:00 PM,003XX N LOTUS AVE,0820,THEFT,$500 AND UNDER,RESIDENCE,False,...,28.0,25.0,06,1139890.0,1901675.0,2016,05/10/2016 03:56:50 PM,41.886297,-87.761751,"(41.886297242, -87.761750709)"


In [ ]:
# 11. 결과를 CSV로 저장
crime_with_community.to_csv('/content/drive/MyDrive/chicago_crime_data_with_ward_community4.csv', index=False)

In [ ]:
crime_with_community[['ID', 'Ward', 'Community Area']]

,ID,Ward,Community Area
0,10508693,24.0,29.0
1,10508695,20.0,42.0
2,10508697,37.0,25.0
3,10508698,28.0,25.0
4,10508699,28.0,25.0
...,...,...,...
1456709,10508679,28.0,30.0
1456710,10508680,17.0,69.0
1456711,10508681,15.0,66.0
1456712,10508690,7.0,46.0
